In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchmetrics
from torchvision import transforms
from torch.utils.data import DataLoader
from torch.utils.data import random_split
import optuna

torch.manual_seed(42)

/opt/anaconda3/envs/ml313/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using {device}")

Using mps


In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.2860,), std=(0.3530,))
])

In [4]:
train_data = torchvision.datasets.FashionMNIST("../data", download=True, train=True, transform=transform)
test_data = torchvision.datasets.FashionMNIST("../data", download=True, train=False, transform=transform)

In [5]:
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

In [6]:
recall_metric = torchmetrics.classification.Recall(task="multiclass", num_classes=10, average="macro").to(device)
precision_metric = torchmetrics.classification.Precision(task="multiclass", num_classes=10, average="macro").to(device)
f1_score_metric = torchmetrics.classification.F1Score(task="multiclass", num_classes=10, average="macro").to(device)

In [7]:
class FlexibleCNN(nn.Module):
    def __init__(self, n_layers, n_filters, kernel_sizes, dropout_rate, num_classes, fc_size):
        super().__init__()

        self.classifier = None
        self.dropout_rate = dropout_rate
        self.num_classes = num_classes
        self.fc_size = fc_size
        self.blocks = nn.ModuleList() # Save model inside ModuleList not just a "plain" Python list 
        in_channels = 1 # gray-scale

        for i in range(n_layers):
            out_channels = n_filters[i]
            kernel_size = kernel_sizes[i]
            padding = (kernel_size - 1) // 2

            block = nn.Sequential(
                nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size, padding=padding),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=2, stride=2)
            )

            self.blocks.append(block)
            in_channels = out_channels

    def _create_classifier(self, flattened_size, device):

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flattened_size, self.fc_size),
            nn.ReLU(),
            nn.Dropout(self.dropout_rate),
            nn.Linear(self.fc_size, self.num_classes)
        ).to(device)

    def forward(self, x):
        device = x.device

        for block in self.blocks:
            x = block(x)

        flattened_layer = torch.flatten(x, 1)
        flattened_size = flattened_layer.shape[1]
        
        if self.classifier is None:
            self._create_classifier(flattened_size, device)

        return self.classifier(x)

In [8]:
def train_epoch(model, train_loader, loss_function, optimizer, device, verbose=False):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == targets).sum().item()
        total += targets.size(0)

        if batch_idx % 100 == 0 and batch_idx > 0:

            accuracy = 100 * correct / total
            if verbose:
                print(f"Loss: {running_loss / 100:.3f} | Train accuracy: {accuracy:.1f}")
            running_loss = 0.0

def evaluate(model, test_loader, device, verbose=False):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == targets).sum().item()
            total += targets.size(0)

            recall_metric.update(preds=predicted, target=targets)
            precision_metric.update(preds=predicted, target=targets)
            f1_score_metric.update(preds=predicted, target=targets)

    recall = recall_metric.compute()
    precision = precision_metric.compute()
    f1_score = f1_score_metric.compute()

    if verbose:
        print(f"Recall: {recall:.3f} | Precision: {precision:.3f} | F1 Score: {f1_score:.3f}")
            
    return 100 * correct / total

In [9]:
def objective(trial, device):
    n_layers = trial.suggest_int("n_layers", 1, 3)
    n_filters = [trial.suggest_int(f"n_filters_{i}", 16, 128) for i in range(n_layers)]
    kernel_sizes = [trial.suggest_int(f"kernel_size_{i}", 3, 5) for i in range(n_layers)]
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)

    model = FlexibleCNN(
        n_layers=n_layers, 
        n_filters=n_filters, 
        kernel_sizes=kernel_sizes, 
        dropout_rate=dropout_rate, 
        num_classes=10, 
        fc_size=128).to(device)

    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    num_epochs = 5
    for epoch in range(num_epochs):
        train_epoch(model, train_loader, loss_function, optimizer, device)
        
    accuracy = evaluate(model, test_loader, device)
    return accuracy

In [12]:
study = optuna.create_study(
    storage="sqlite:///../data/study/flexCNN_no_scheduler.db",
    study_name="flexCNN_no_scheduler", 
    direction="maximize", 
    load_if_exists=True)

study.optimize(lambda trial: objective(trial, device), n_trials=5)

[I 2026-08-12 20:49:38,149] Using an existing study with name 'flexCNN_no_scheduler' instead of creating a new one.
[I 2026-08-12 20:51:06,561] Trial 5 finished with value: 90.29 and parameters: {'n_layers': 3, 'n_filters_0': 77, 'n_filters_1': 71, 'n_filters_2': 88, 'kernel_size_0': 5, 'kernel_size_1': 4, 'kernel_size_2': 3, 'dropout_rate': 0.13617664139260938}. Best is trial 5 with value: 90.29.
[I 2026-08-12 20:52:04,448] Trial 6 finished with value: 87.94 and parameters: {'n_layers': 2, 'n_filters_0': 33, 'n_filters_1': 90, 'kernel_size_0': 4, 'kernel_size_1': 5, 'dropout_rate': 0.3807560725642231}. Best is trial 5 with value: 90.29.
[I 2026-08-12 20:52:50,001] Trial 7 finished with value: 55.72 and parameters: {'n_layers': 1, 'n_filters_0': 27, 'kernel_size_0': 4, 'dropout_rate': 0.45373845270693863}. Best is trial 5 with value: 90.29.
[I 2026-08-12 20:54:04,322] Trial 8 finished with value: 88.02 and parameters: {'n_layers': 2, 'n_filters_0': 32, 'n_filters_1': 63, 'kernel_size_0

In [13]:
optuna.visualization.plot_optimization_history(study)

In [14]:
optuna.visualization.plot_param_importances(study)

In [15]:
optuna.visualization.plot_parallel_coordinate(study)

In [23]:
best_params = study.best_params
params_dict = {"n_layers": best_params["n_layers"], 
               "n_filters": [], 
               "kernel_sizes": [], 
               "dropout_rate": best_params["dropout_rate"]}

for param in best_params:

    if param.startswith("n_filters"):
        params_dict["n_filters"].append(best_params[param])
    elif param.startswith("kernel_size"):
        params_dict["kernel_sizes"].append(best_params[param])

params_dict

{'n_layers': 3,
 'n_filters': [77, 71, 88],
 'kernel_sizes': [5, 4, 3],
 'dropout_rate': 0.13617664139260938}

In [24]:
model = FlexibleCNN(
    n_layers=params_dict["n_layers"],
    n_filters=params_dict["n_filters"],
    kernel_sizes=params_dict["kernel_sizes"],
    dropout_rate=params_dict["dropout_rate"],
    num_classes=10,
    fc_size=129
).to(device)

num_epochs = 10
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(num_epochs):
    print(f"Epoch {epoch}")
    train_epoch(model, train_loader, loss_function, optimizer, device, verbose=True)
    accuracy = evaluate(model, test_loader, device, verbose=True)
    print(f"Test accuracy: {accuracy:.1f}")

Epoch 0
Loss: 1.117 | Train accuracy: 61.1
Loss: 0.643 | Train accuracy: 69.0
Loss: 0.566 | Train accuracy: 72.5
Loss: 0.509 | Train accuracy: 74.8
Loss: 0.468 | Train accuracy: 76.5
Loss: 0.442 | Train accuracy: 77.7
Loss: 0.429 | Train accuracy: 78.7
Loss: 0.432 | Train accuracy: 79.5
Loss: 0.398 | Train accuracy: 80.2
Recall: 0.787 | Precision: 0.789 | F1 Score: 0.787
Test accuracy: 86.7
Epoch 1
Loss: 0.371 | Train accuracy: 87.0
Loss: 0.373 | Train accuracy: 87.1
Loss: 0.372 | Train accuracy: 86.8
Loss: 0.367 | Train accuracy: 86.9
Loss: 0.347 | Train accuracy: 87.0
Loss: 0.342 | Train accuracy: 87.1
Loss: 0.337 | Train accuracy: 87.3
Loss: 0.344 | Train accuracy: 87.3
Loss: 0.342 | Train accuracy: 87.3
Recall: 0.801 | Precision: 0.802 | F1 Score: 0.800
Test accuracy: 88.2
Epoch 2
Loss: 0.316 | Train accuracy: 89.0
Loss: 0.295 | Train accuracy: 89.1
Loss: 0.311 | Train accuracy: 88.9
Loss: 0.314 | Train accuracy: 88.9
Loss: 0.317 | Train accuracy: 88.8
Loss: 0.303 | Train accuracy:

In [27]:
# With scheduler
model = FlexibleCNN(
    n_layers=params_dict["n_layers"],
    n_filters=params_dict["n_filters"],
    kernel_sizes=params_dict["kernel_sizes"],
    dropout_rate=params_dict["dropout_rate"],
    num_classes=10,
    fc_size=129
).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=0)

for epoch in range(num_epochs):
    print(f"Epoch {epoch}")
    train_epoch(model, train_loader, loss_function, optimizer, device, verbose=True)
    accuracy = evaluate(model, test_loader, device, verbose=True)
    scheduler.step()
    print(f"Accuracy: {accuracy:.3f}")

Epoch 0
Loss: 1.121 | Train accuracy: 61.8
Loss: 0.678 | Train accuracy: 68.7
Loss: 0.561 | Train accuracy: 72.5
Loss: 0.526 | Train accuracy: 74.5
Loss: 0.472 | Train accuracy: 76.3
Loss: 0.465 | Train accuracy: 77.5
Loss: 0.460 | Train accuracy: 78.3
Loss: 0.412 | Train accuracy: 79.1
Loss: 0.416 | Train accuracy: 79.8
Recall: 0.868 | Precision: 0.867 | F1 Score: 0.867
Accuracy: 86.540
Epoch 1
Loss: 0.400 | Train accuracy: 86.0
Loss: 0.380 | Train accuracy: 86.0
Loss: 0.364 | Train accuracy: 86.5
Loss: 0.349 | Train accuracy: 86.8
Loss: 0.369 | Train accuracy: 86.8
Loss: 0.363 | Train accuracy: 86.9
Loss: 0.343 | Train accuracy: 87.0
Loss: 0.326 | Train accuracy: 87.2
Loss: 0.331 | Train accuracy: 87.3
Recall: 0.869 | Precision: 0.868 | F1 Score: 0.868
Accuracy: 88.530
Epoch 2
Loss: 0.326 | Train accuracy: 88.2
Loss: 0.322 | Train accuracy: 88.3
Loss: 0.311 | Train accuracy: 88.4
Loss: 0.315 | Train accuracy: 88.5
Loss: 0.311 | Train accuracy: 88.6
Loss: 0.291 | Train accuracy: 88.8
